# 7. BM25 & Rerank

This topic covers two important search tools used in RAG:

1. **BM25** → a classic **keyword search** method.
2. **Rerank** → a step that **re-orders** search results so the **best ones come first**.

## Part 1: What is BM25?

**BM25** is a popular formula that finds documents by **matching keywords**.

It gives a higher score to a document when:
- The search words appear **more often** in it.
- The words are **rare** (special words matter more than common words like "the").

It is great for matching **exact terms** like names, IDs, or specific words.

## BM25 vs Embeddings (Semantic search)

| | BM25 (keyword) | Embeddings (semantic) |
|--|----------------|------------------------|
| Matches | Exact **words** | **Meaning** |
| Good for | Names, codes, exact terms | Similar ideas, synonyms |
| Example | finds "iPhone 15" exactly | knows "phone" ≈ "mobile" |

Many RAG systems use **both** together (called **Hybrid Search**).

## BM25 in code

Install the small library first (in the terminal):
```
pip install rank-bm25
```

In [ ]:
from rank_bm25 import BM25Okapi

# Our documents (each split into words)
docs = [
    "Python is a programming language",
    "Streamlit builds web apps",
    "Python is used for AI and machine learning"
]
tokenized_docs = [doc.lower().split() for doc in docs]

# Build the BM25 search
bm25 = BM25Okapi(tokenized_docs)

# Search
query = "python ai".split()
scores = bm25.get_scores(query)

for doc, score in zip(docs, scores):
    print(round(score, 2), "->", doc)

## Part 2: What is Rerank?

After we search (with BM25 or embeddings), we get a list of documents.

But the **best one may not be at the very top**. A **Reranker** looks at the question and each result **together**, and **re-orders** them so the most relevant one comes first.

```
Search results (rough order)
        |
        v
   Reranker looks at question + each doc
        |
        v
Better order (most relevant on top)
```

## Why rerank?

- The first search is **fast** but not perfect.
- The reranker is **slower** but **more accurate**.
- So we do it in 2 steps:
  1. **Search** → quickly get the top 20 documents.
  2. **Rerank** → carefully pick the best 3 from those 20.

This gives both **speed** and **accuracy**.

## A simple rerank idea (pseudo-code)

A real reranker uses an AI model (like a Cross-Encoder), but the idea is simple: give each document a relevance score, then sort.

In [ ]:
question = "python for ai"
results = [
    "Streamlit builds web apps",
    "Python is used for AI and machine learning",
    "Python is a programming language"
]

# Simple score: count how many question words appear in the doc
def rerank_score(question, doc):
    q_words = question.lower().split()
    return sum(1 for w in q_words if w in doc.lower())

reranked = sorted(results, key=lambda d: rerank_score(question, d), reverse=True)

print("Reranked results (best first):")
for d in reranked:
    print("-", d)

## Key points to remember

- **BM25** = keyword search (matches exact words).
- **Embeddings** = semantic search (matches meaning).
- Using both = **Hybrid Search**.
- **Rerank** = re-order results so the **best ones come first**.
- Common pattern: **search many ➜ rerank ➜ keep the top few**.
- This gives both **speed** and **accuracy**.